# ARTIQ Non-NDScan Parameter Types

This notebook explores the standard ARTIQ argument types that exist alongside or instead of ndscan_params.
These are the classic ARTIQ arginfo types used for experiment configuration.

In [16]:
import json
import pprint
from collections import Counter, defaultdict

# Load experiments
explist = json.load(open("explist_debug.json", "r"))
experiments = explist["experiments"]

print(f"Total experiments: {len(experiments)}")

Total experiments: 161


## 1. Arginfo Structure

Each parameter in `arginfo` is a tuple: `[type_dict, group, tooltip]`

In [17]:
# Collect all non-ndscan parameters
all_params = []
for exp in experiments:
    arginfo = exp.get("arginfo", {})
    for param_name, param_tuple in arginfo.items():
        if param_name == "ndscan_params":
            continue
        if param_tuple[0] is not None:
            all_params.append(
                {
                    "experiment": exp["name"],
                    "param_name": param_name,
                    "type_dict": param_tuple[0],
                    "group": param_tuple[1],
                    "tooltip": param_tuple[2],
                }
            )

print(f"Total non-ndscan parameters: {len(all_params)}")

# Count by type
type_counts = Counter(p["type_dict"].get("ty", "unknown") for p in all_params)
print("\nParameter types:")
for ty, count in type_counts.most_common():
    print(f"  {ty}: {count}")

Total non-ndscan parameters: 138

Parameter types:
  NumberValue: 61
  BooleanValue: 51
  EnumerationValue: 21
  StringValue: 5


## 2. EnumerationValue Type

Used for dropdown selections (e.g., device/channel selection)

In [18]:
# Collect EnumerationValue examples
enum_params = [p for p in all_params if p["type_dict"].get("ty") == "EnumerationValue"]
print(f"EnumerationValue parameters: {len(enum_params)}")

# Show structure
print("\nExample EnumerationValue:")
pprint.pprint(enum_params[0])

EnumerationValue parameters: 21

Example EnumerationValue:
{'experiment': 'injected_diodes/Single relocker board channel',
 'group': None,
 'param_name': 'channel_name',
 'tooltip': None,
 'type_dict': {'choices': ['blue_IJD1_relocker',
                           'blue_IJD2_relocker',
                           'blue_IJD3_relocker',
                           'red_IJD1_relocker'],
               'default': 'blue_IJD1_relocker',
               'quickstyle': False,
               'ty': 'EnumerationValue'}}


In [19]:
# Analyze choices
print("EnumerationValue fields:")
enum_fields = Counter()
for p in enum_params:
    for key in p["type_dict"].keys():
        enum_fields[key] += 1

for field, count in enum_fields.most_common():
    pct = count / len(enum_params) * 100
    print(f"  {field}: {count} ({pct:.1f}%)")

EnumerationValue fields:
  ty: 21 (100.0%)
  choices: 21 (100.0%)
  quickstyle: 21 (100.0%)
  default: 17 (81.0%)


In [20]:
# Analyze number of choices
choice_counts = [len(p["type_dict"].get("choices", [])) for p in enum_params]
print(
    f"Choices count: min={min(choice_counts)}, max={max(choice_counts)}, avg={sum(choice_counts) / len(choice_counts):.1f}"
)

# Show some example choice sets
print("\nExample choice sets:")
seen = set()
for p in enum_params:
    choices = tuple(p["type_dict"].get("choices", [])[:5])  # First 5
    if choices not in seen:
        seen.add(choices)
        print(f"\n  {p['param_name']} ({len(p['type_dict'].get('choices', []))} choices):")
        for c in choices:
            print(f"    - {c}")
    if len(seen) >= 5:
        break

Choices count: min=3, max=48, avg=20.5

Example choice sets:

  channel_name (4 choices):
    - blue_IJD1_relocker
    - blue_IJD2_relocker
    - blue_IJD3_relocker
    - red_IJD1_relocker

  channel_name (5 choices):
    - blue_IJD1_relocker
    - blue_IJD2_relocker
    - blue_IJD3_relocker
    - red_IJD1_relocker
    - filter_cavity_scanner

  controller_name (4 choices):
    - blue_IJD1_controller
    - blue_IJD2_controller
    - blue_IJD3_controller
    - red_IJD1_controller

  device_name (23 choices):
    - urukul9910_aom_doublepass_461_master_to_ijd1
    - urukul9910_aom_singlepass_461_ijd1_to_ijd23
    - urukul9910_aom_doublepass_461_to_xfer_cavity
    - urukul9912_aom_singlepass_461_imaging_switch
    - urukul9910_aom_doublepass_689_red_injection

  current_supply (4 choices):
    - chamber_2_coil_x
    - chamber_2_coil_y
    - chamber_2_coil_z
    - chamber_2_coil_mot


## 3. BooleanValue Type

In [21]:
# Collect BooleanValue examples
bool_params = [p for p in all_params if p["type_dict"].get("ty") == "BooleanValue"]
print(f"BooleanValue parameters: {len(bool_params)}")

# Show structure
print("\nExample BooleanValue:")
pprint.pprint(bool_params[0])

# Analyze fields
print("\nBooleanValue fields:")
bool_fields = Counter()
for p in bool_params:
    for key in p["type_dict"].keys():
        bool_fields[key] += 1

for field, count in bool_fields.most_common():
    print(f"  {field}: {count}")

BooleanValue parameters: 51

Example BooleanValue:
{'experiment': 'utilities/Enable or disable the current for the Toptica lasers',
 'group': None,
 'param_name': 'enable_laser_currents',
 'tooltip': 'For the lasers being controlled, enable the current?',
 'type_dict': {'default': False, 'ty': 'BooleanValue'}}

BooleanValue fields:
  ty: 51
  default: 51


In [22]:
# Analyze default values
true_count = sum(1 for p in bool_params if p["type_dict"].get("default") is True)
false_count = sum(1 for p in bool_params if p["type_dict"].get("default") is False)
print(f"Default True: {true_count}")
print(f"Default False: {false_count}")

# Show example parameter names
print("\nExample BooleanValue parameter names:")
for p in bool_params[:10]:
    default = p["type_dict"].get("default")
    print(f"  {p['param_name']}: default={default}")

Default True: 37
Default False: 14

Example BooleanValue parameter names:
  enable_laser_currents: default=False
  control_toptica_461: default=False
  control_toptica_679: default=False
  control_toptica_1379: default=False
  control_toptica_698: default=False
  control_toptica_707: default=False
  control_toptica_689: default=False
  control_toptica_487: default=False
  control_toptica_641: default=False
  switch: default=True


## 4. NumberValue Type

In [23]:
# Collect NumberValue examples
num_params = [p for p in all_params if p["type_dict"].get("ty") == "NumberValue"]
print(f"NumberValue parameters: {len(num_params)}")

# Show structure
print("\nExample NumberValue:")
pprint.pprint(num_params[0])

# Analyze fields
print("\nNumberValue fields:")
num_fields = Counter()
for p in num_params:
    for key in p["type_dict"].keys():
        num_fields[key] += 1

for field, count in num_fields.most_common():
    pct = count / len(num_params) * 100
    print(f"  {field}: {count} ({pct:.1f}%)")

NumberValue parameters: 61

Example NumberValue:
{'experiment': 'utilities/Basic DDS setter for AD9910s or AD9912s',
 'group': None,
 'param_name': 'frequency',
 'tooltip': None,
 'type_dict': {'default': 100000000.0,
               'max': None,
               'min': None,
               'precision': 2,
               'scale': 1000000.0,
               'step': 100000.0,
               'ty': 'NumberValue',
               'type': 'auto',
               'unit': 'MHz'}}

NumberValue fields:
  ty: 61 (100.0%)
  unit: 61 (100.0%)
  scale: 61 (100.0%)
  step: 61 (100.0%)
  min: 61 (100.0%)
  max: 61 (100.0%)
  precision: 61 (100.0%)
  type: 61 (100.0%)
  default: 60 (98.4%)


In [24]:
# Analyze by 'type' subfield (int vs float)
num_subtypes = Counter(p["type_dict"].get("type", "auto") for p in num_params)
print("NumberValue subtypes:")
for subtype, count in num_subtypes.most_common():
    print(f"  {subtype}: {count}")

# Analyze units
units = Counter(p["type_dict"].get("unit", "(none)") for p in num_params)
print("\nUnits:")
for unit, count in units.most_common()[:10]:
    print(f"  {unit}: {count}")

NumberValue subtypes:
  auto: 30
  float: 16
  int: 15

Units:
  : 32
  MHz: 16
  us: 4
  dB: 3
  s: 3
  V: 2
  ms: 1


In [25]:
# Examples with different configurations
print("=== NumberValue variations ===")

# With min/max
with_bounds = [p for p in num_params if p["type_dict"].get("min") is not None and p["type_dict"].get("max") is not None]
print(f"\nWith min and max: {len(with_bounds)}")
if with_bounds:
    p = with_bounds[0]
    print(f"  Example: {p['param_name']}")
    print(f"    min={p['type_dict']['min']}, max={p['type_dict']['max']}")

# Int type
int_nums = [p for p in num_params if p["type_dict"].get("type") == "int"]
print(f"\nInt type: {len(int_nums)}")
if int_nums:
    p = int_nums[0]
    print(f"  Example: {p['param_name']}")
    pprint.pprint(p["type_dict"])

=== NumberValue variations ===

With min and max: 3
  Example: attenuation
    min=0, max=30

Int type: 15
  Example: adc_channel
{'default': 0,
 'max': None,
 'min': None,
 'precision': 0,
 'scale': 1,
 'step': 1,
 'ty': 'NumberValue',
 'type': 'int',
 'unit': ''}


## 5. StringValue Type

In [26]:
# Collect StringValue examples
str_params = [p for p in all_params if p["type_dict"].get("ty") == "StringValue"]
print(f"StringValue parameters: {len(str_params)}")

if str_params:
    print("\nExample StringValue:")
    pprint.pprint(str_params[0])

    print("\nAll StringValue parameters:")
    for p in str_params:
        print(f'  {p["experiment"]}: {p["param_name"]} = "{p["type_dict"].get("default", "")}"')

StringValue parameters: 5

Example StringValue:
{'experiment': 'utilities/Take a reading from a sampler',
 'group': None,
 'param_name': 'sampler_device_name',
 'tooltip': 'Sampler device to read',
 'type_dict': {'default': 'sampler2', 'ty': 'StringValue'}}

All StringValue parameters:
  utilities/Take a reading from a sampler: sampler_device_name = "sampler2"
  tests/TestAD9910LaneUsage: urukul_channel = "urukul8_ch2"
  tests/TestSUServoAttenuationReading: channel_name = "suservo_aom_singlepass_689_red_mot_diagonal"
  tests/Determine servo period: Servo device name = "suservo0"
  tests/ToggleTTL: ttl_device = ""


## 6. PYONValue Type (for ndscan_params)

This is the type used for the ndscan_params itself.

In [27]:
# Check the structure of PYONValue in arginfo
for exp in experiments:
    arginfo = exp.get("arginfo", {})
    if "ndscan_params" in arginfo:
        print("ndscan_params arginfo tuple structure:")
        ndscan_raw = arginfo["ndscan_params"]
        print(f"  [0] type_dict: {type(ndscan_raw[0]).__name__}")
        if ndscan_raw[0]:
            print(f"      keys: {list(ndscan_raw[0].keys())}")
            print(f"      ty: {ndscan_raw[0].get('ty')}")
        print(f"  [1] group: {ndscan_raw[1]}")
        print(f"  [2] tooltip: {ndscan_raw[2]}")
        break

ndscan_params arginfo tuple structure:
  [0] type_dict: dict
      keys: ['ty', 'default']
      ty: PYONValue
  [1] group: None
  [2] tooltip: None


## 7. Parameter Groups

In [28]:
# Analyze parameter groups
groups = Counter(p["group"] for p in all_params)
print("Parameter groups:")
for group, count in groups.most_common():
    display = group if group else "(no group)"
    print(f"  {display}: {count}")

Parameter groups:
  (no group): 118
  Lasers: 8
  461: 2
  689: 2
  707: 2
  679: 2
  688: 2
  487: 2


In [29]:
# Show example with groups
grouped_exp = None
for exp in experiments:
    arginfo = exp.get("arginfo", {})
    groups = set()
    for param_name, param_tuple in arginfo.items():
        if param_tuple[1]:  # Has a group
            groups.add(param_tuple[1])
    if len(groups) > 1:
        grouped_exp = exp
        break

if grouped_exp:
    print(f"Experiment with multiple groups: {grouped_exp['name']}\n")
    by_group = defaultdict(list)
    for param_name, param_tuple in grouped_exp["arginfo"].items():
        group = param_tuple[1] or "(no group)"
        by_group[group].append(param_name)

    for group, params in sorted(by_group.items()):
        print(f"  {group}:")
        for p in params:
            print(f"    - {p}")

Experiment with multiple groups: utilities/Enable or disable wavemeter locking with WAND

  461:
    - 461_enabled
    - 461_offset
  487:
    - 487_enabled
    - 487_offset
  679:
    - 679_enabled
    - 679_offset
  688:
    - 688_enabled
    - 688_offset
  689:
    - 689_enabled
    - 689_offset
  707:
    - 707_enabled
    - 707_offset


## Summary: ARTIQ Arginfo Types

In [30]:
print("""
=============================================================================
ARTIQ Argument Types Summary
=============================================================================

Arginfo Tuple Format: [type_dict, group, tooltip]

Type Dictionary Fields by Type:

1. EnumerationValue
   - ty: "EnumerationValue"
   - default: string (one of choices)
   - choices: list of strings
   - quickstyle: bool (optional)

2. BooleanValue
   - ty: "BooleanValue"
   - default: bool

3. NumberValue
   - ty: "NumberValue"
   - default: number
   - unit: string (optional, e.g., "MHz", "dB")
   - scale: number (conversion factor)
   - step: number (UI increment)
   - min: number (optional)
   - max: number (optional)
   - precision: int (decimal places)
   - type: "auto" | "int" | "float"

4. StringValue
   - ty: "StringValue"
   - default: string

5. PYONValue (used for ndscan_params)
   - ty: "PYONValue"
   - default: PYON-encoded string

=============================================================================
""")


ARTIQ Argument Types Summary

Arginfo Tuple Format: [type_dict, group, tooltip]

Type Dictionary Fields by Type:

1. EnumerationValue
   - ty: "EnumerationValue"
   - default: string (one of choices)
   - choices: list of strings
   - quickstyle: bool (optional)

2. BooleanValue
   - ty: "BooleanValue"
   - default: bool

3. NumberValue
   - ty: "NumberValue"
   - default: number
   - unit: string (optional, e.g., "MHz", "dB")
   - scale: number (conversion factor)
   - step: number (UI increment)
   - min: number (optional)
   - max: number (optional)
   - precision: int (decimal places)
   - type: "auto" | "int" | "float"

4. StringValue
   - ty: "StringValue"
   - default: string

5. PYONValue (used for ndscan_params)
   - ty: "PYONValue"
   - default: PYON-encoded string


